In [ ]:
using CSV, DataFrames, Flux, Statistics

In [ ]:
path_file_train = "../data/dataModelCSV/test_data.csv"
path_file_test = "../data/dataModelCSV/train_data.csv"
data_train = CSV.read(path_file_train, DataFrame; delim=';')
data_test = CSV.read(path_file_test, DataFrame; delim=';')
first(data_train, 3)

In [ ]:
first(data_test, 3)

In [ ]:
column_name = names(data_train)

In [ ]:
# 1. Preparaci?n de datos
# Separar features y target
X_train = select(data_train, Not(:GRD))
y_train = data_train.GRD
X_test = select(data_test, Not(:GRD))
y_test = data_test.GRD



In [ ]:
# Convertir datos categ?ricos a num?ricos
# Primero identificamos columnas categ?ricas (excepto GRD que es nuestro target)
cat_cols = names(X_train, AbstractString)
num_cols = names(X_train, Union{Int, Float64})
num_cols = filter(col -> col != "Edad en a?os", num_cols)  # Asumiendo que Edad es num?rica



In [ ]:
# One-hot encoding para variables categ?ricas
using MLJ: StandardScaler, fit!, transform, machine
hot = OneHotEncoder()
X_cat_train = MLJ.transform(fit!(machine(hot, X_train[!, cat_cols])), X_train[!, cat_cols])
X_cat_test = MLJ.transform(fit!(machine(hot, X_test[!, cat_cols])), X_test[!, cat_cols])




In [ ]:
# Normalizar variables num?ricas si existen
if !isempty(num_cols)
    scaler = StandardScaler()
    X_num_train = MLJ.transform(fit!(machine(scaler, X_train[!, num_cols])), X_train[!, num_cols])
    X_num_test = MLJ.transform(fit!(machine(scaler, X_test[!, num_cols])), X_test[!, num_cols])
    
    # Combinar datos categ?ricos y num?ricos
    X_train_processed = hcat(X_cat_train, X_num_train)
    X_test_processed = hcat(X_cat_test, X_num_test)
else
    X_train_processed = X_cat_train
    X_test_processed = X_cat_test
end



In [ ]:
# Para la variable objetivo, necesitamos convertirla a one-hot tambi?n
# Primero encontramos todas las clases ?nicas
grd_classes = unique(vcat(y_train, y_test))
num_classes = length(grd_classes)

# Crear un diccionario para mapear GRD a ?ndices
grd_to_idx = Dict(class => i for (i, class) in enumerate(grd_classes))

# Convertir etiquetas a ?ndices
y_train_idx = [grd_to_idx[grd] for grd in y_train]
y_test_idx = [grd_to_idx[grd] for grd in y_test]

# Convertir a one-hot encoding
y_train_onehot = Flux.onehotbatch(y_train_idx, 1:num_classes)
y_test_onehot = Flux.onehotbatch(y_test_idx, 1:num_classes)

# Preparar datos para Flux
X_train_matrix = Matrix(X_train_processed)'  # Transpuesta para formato Flux
X_test_matrix = Matrix(X_test_processed)'
train_data = [(X_train_matrix, y_train_onehot)]
test_data = [(X_test_matrix, y_test_onehot)]

# 2. Definir la arquitectura de la red neuronal
input_size = size(X_train_matrix, 1)
hidden_size = 128  # Ajusta seg?n la complejidad de tus datos
output_size = num_classes

model = Chain(
    Dense(input_size, hidden_size, relu),
    Dropout(0.5),  # Para prevenir overfitting
    Dense(hidden_size, hidden_size?2, relu),
    Dropout(0.3),
    Dense(hidden_size?2, output_size),
    softmax
)

# 3. Definir la funci?n de p?rdida y el optimizador
loss(x, y) = Flux.crossentropy(model(x), y)
opt = ADAM(0.001)

# 4. Entrenamiento
epochs = 50
batch_size = 32

# Crear minibatches
train_loader = Flux.DataLoader((X_train_matrix, y_train_onehot), batchsize=batch_size, shuffle=true)

# Funci?n para evaluar el modelo
function evaluate(x, y)
    ? = model(x)
    accuracy = mean(Flux.onecold(?) .== Flux.onecold(y))
    return accuracy
end

# Bucle de entrenamiento
for epoch in 1:epochs
    Flux.train!(loss, Flux.params(model), train_loader, opt)
    
    # Calcular accuracy en cada ?poca
    if epoch % 5 == 0
        train_acc = evaluate(X_train_matrix, y_train_onehot)
        test_acc = evaluate(X_test_matrix, y_test_onehot)
        println("Epoch $epoch: Train accuracy = $train_acc, Test accuracy = $test_acc")
    end
end

# 5. Evaluaci?n del modelo
final_train_acc = evaluate(X_train_matrix, y_train_onehot)
final_test_acc = evaluate(X_test_matrix, y_test_onehot)
println("Final results: Train accuracy = $final_train_acc, Test accuracy = $final_test_acc")

# Para hacer predicciones con nuevos datos
function predict_grd(new_data)
    # Preprocesar los nuevos datos igual que los de entrenamiento
    # ...
    
    # Hacer la predicci?n
    probabilities = model(new_data_processed)
    predicted_idx = Flux.onecold(probabilities)
    predicted_grd = [grd_classes[idx] for idx in predicted_idx]
    
    return predicted_grd
end